In [ ]:
# NO recommended / NO xyimb — S+K+MK mix being tuned in skmk_aut_tune.
# READY — see results/heuristic_search/scale10k/COLAB_RECOMMENDATION.md
# ===== COLAB 5×51GB HIGH-SPEEDUP — CONFIG (edit ONLY CHUNK_INDEX if needed) ==
# Five Colab High-RAM (~51 GB) sessions in parallel. Each session runs ONE
# stride chunk (CHUNKS=5). Inside a session, N_WORKERS=auto uses hcompact
# memory sizing (~6 workers typical at budget 200k on 51 GB).
#
# After wall: see results/heuristic_search/scale10k/COLAB_RECOMMENDATION.md
# Hand Drive jsonls back; merge with merge_colab_chunks.

REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "ACSolverX"
BRANCH     = "cursor/heur-12h-anti-overfit-a42e"
CLONE      = True
UPDATE_REPO = True

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx/hsearch_colab5"  # NEVER /content/drive alone

# ---- THIS SESSION'S SHARD (1..5). Pre-set per notebook file. --------------
CHUNK_INDEX = 5

cfg = dict(
    DATASET   = "unsolved124",
    SUBSET    = None,

    # Will be stamped after wall; safe interim defaults from scale10k so far:
    ARMS      = ['baseline', 's12', 's28'],  # no recommended/xyimb; mix TBD from skmk_aut_tune

    CHUNKS       = 5,
    CHUNK_INDEX  = CHUNK_INDEX,
    # High-speedup: hcompact + auto workers (memory-capped for 51 GB).
    N_WORKERS    = "auto",

    NODE_BUDGET = 200_000,
    CHECKPOINTS = [1000, 5000, 10000, 25000, 50000, 100000, 200000],

    MAX_RELATOR_LENGTH = 64,
    ENGINE    = "hcompact",
    KEEP_PATH = True,
    RESUME    = True,
    OUT_STEM  = "hsearch_colab5",
    # Local stage dir (not Drive) — run_ab mirrors whole-file every 60s.
    STAGE_DIR = "/content/hsearch_stage",
)

HEARTBEAT_SECS = 60
PROGRESS_SECS  = 300
    ARMS      = ['baseline', 's12', 's28'],  # no recommended/xyimb; mix TBD from skmk_aut_tune


In [ ]:
# ==================== SETUP (clone / pull / mount / purge) ================
import os, sys, subprocess, importlib

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
    os.makedirs(cfg.get("STAGE_DIR", "/content/hsearch_stage"), exist_ok=True)
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from experiments.heuristic_search.core.hsolve import greedy_search_h
from experiments.heuristic_search.runners import run_s24_scale as _rs
_ = greedy_search_h("xyx", "yx", 20, max_relator_length=32,
                    config=_rs.run_ab.ARMS["s20"])
print("kernels warm — setup done")
print("tip: Runtime → Restart → Run All resumes (UPDATE_REPO + flock + RESUME)")


In [ ]:
# ==================== RUN (high-speedup multi-worker) =====================
from experiments.heuristic_search.runners.run_s24_scale import run_s24_scale
import os
print(f"N_WORKERS={cfg['N_WORKERS']} ENGINE={cfg['ENGINE']} "
      f"budget={cfg['NODE_BUDGET']:,} chunk={cfg['CHUNK_INDEX']}/{cfg['CHUNKS']}")
run_s24_scale(
    cfg,
    out_dir=(DRIVE_DIR if (IN_COLAB and MOUNT_DRIVE) else "results/hsearch"),
    heartbeat_secs=HEARTBEAT_SECS,
    progress_secs=PROGRESS_SECS,
)
print("done — leave this session up until the jsonl mirror finishes; "
      "then open the next chunk notebook or shut down.")
